# 使用GPU加速PyTorch计算


- **高计算需求**：
  - 模型参数量大，需要并行计算  
  - 矩阵运算（如：`矩阵乘法@`）占模型计算量90%+  
  - 数据集规模大
- **显卡加速优势**：  
  - GPU含数千计算核心 
  - 专为并行计算设计 
  - 高带宽显存，适用高速数据吞吐
- **CUDA**：  
  - NVIDIA开发的通用并行计算架构
  - 提供`核函数-流处理器`映射机制  
  - 广泛应用，社区支持强大

## 1. 检查CUDA可用性

在开始前需要确认CUDA设备是否可用

In [1]:
import torch

# 检查CUDA是否可用
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Available devices: {torch.cuda.device_count()}")  # 可用GPU数量
    print(f"Current device: {torch.cuda.current_device()}")   # 当前设备索引
    print(f"Device name: {torch.cuda.get_device_name(0)}")    # 获取第一个GPU名称

CUDA available: True
Available devices: 1
Current device: 0
Device name: NVIDIA A100-SXM4-80GB


## 2. 设置device


In [2]:
# 定义全局device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 之后可以将模型和数据移动到device上

Using device: cuda


### 2.1 Tensor device

In [3]:
# 创建CPU张量
x = torch.randn(1000, 1000)
print(f"Original device: {x.device}")

# 转移到GPU（两种方法）
x_gpu = x.cuda()            # 默认使用cuda:0
x_gpu = x.to(device='cuda:0')  # 明确指定设备

# 检查设备类型
print(f"GPU tensor device: {x_gpu.device}")

# 移回CPU
x_cpu = x_gpu.cpu()

Original device: cpu
GPU tensor device: cuda:0


### 2.2 Model device


In [4]:
from torchvision.models import resnet18

model = resnet18(weights='IMAGENET1K_V1')
model = model.cuda()  # 将整个模型参数转移到GPU

# 验证模型参数设备
next(model.parameters()).device  # 应显示cuda:x

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100.0%


device(type='cuda', index=0)

## 3. GPU加速测试

### 3.1 基本张量运算

In [5]:
# 创建两个大型随机矩阵
a = torch.randn(10000, 10000).cuda()
b = torch.randn(10000, 10000).cuda()

# GPU矩阵乘法（自动使用CUDA加速）
c = torch.matmul(a, b)

c

tensor([[ -38.3277,  -43.3470,  -23.3908,  ..., -122.4506,   17.1426,
          -65.0747],
        [ -76.4496,  -40.6365,   -3.3200,  ...,  330.1714,  208.7293,
          -36.2943],
        [  -3.1703,  -57.2339,  -18.7697,  ...,   48.7078,    5.3865,
            9.0708],
        ...,
        [-106.0629,  -12.0988,   49.7078,  ..., -253.1612,    3.9701,
          -44.5663],
        [  91.9841,   -4.3721,  102.0486,  ...,  164.9751,    0.8036,
          -79.7257],
        [ -67.1310, -153.4147,   69.2297,  ...,   38.8682,  225.1280,
            1.3552]], device='cuda:0')

In [6]:
# 性能对比测试
def benchmark():
    # CPU版本
    a_cpu = torch.randn(1000, 1000)
    b_cpu = torch.randn(1000, 1000)
    
    # GPU版本
    a_gpu = a_cpu.cuda()
    b_gpu = b_cpu.cuda()
    
    # 计时
    torch.cuda.synchronize()
    %timeit torch.matmul(a_cpu, b_cpu)  # CPU时间
    
    torch.cuda.synchronize()
    %timeit torch.matmul(a_gpu, b_gpu)  # GPU时间
    
benchmark()

90 ms ± 7.54 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
124 μs ± 15 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### 3.2 模型训练示例

In [7]:
import torch.nn as nn
import torch.optim as optim
from torchvision.transforms import ToTensor
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

# 配置参数
batch_size = 256
num_workers = 0

# 数据加载
# 注意：数据默认在CPU上
train_set = CIFAR10(root='./data', train=True, transform=ToTensor(), download=False)
train_loader = DataLoader(train_set, batch_size=batch_size, 
                         shuffle=True, num_workers=num_workers)

# 定义简单模型
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 16, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Linear(32, 10)
    
    def forward(self, x):
        x = self.conv_layers(x)
        return self.classifier(x.flatten(1))

model = SimpleCNN().cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = optim.Adam(model.parameters())

# 训练
for epoch in range(3):
    for images, labels in train_loader:
        # 关键步骤：将数据转移到GPU
        images = images.cuda().float()  
        labels = labels.cuda()
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    
    print(f'Epoch {epoch+1} loss: {loss.item():.4f}')

Epoch 1 loss: 2.0580
Epoch 2 loss: 1.9362
Epoch 3 loss: 1.9392


## 4. 多GPU并行计算（数据并行）（了解）

In [ ]:
# 使用DataParallel实现单机多卡
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    parallel_model = nn.DataParallel(model)
else:
    print("Single GPU available")

## 5. 注意事项

### 5.1 设备一致性

**所有参与计算的张量必须在同一设备上**

In [8]:
# 错误示例

try:
    a = torch.randn(10).cuda()
    b = torch.randn(10).cpu()
    c = a + b  # 会抛出错误
except RuntimeError as e:
    print("Error:", e)

Error: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!


### 5.2 GPU显存管理

In [9]:
# 监控显存使用：
print(torch.cuda.memory_allocated(0)/1024**2, "MB used")  # 当前显存使用量
print(torch.cuda.max_memory_allocated(0)/1024**2, "MB peak")  # 峰值使用量

# 清空缓存：
torch.cuda.empty_cache()


410.7275390625 MB used
1221.50927734375 MB peak


常用指令：`nvidia-smi`

用于查看NVIDIA显卡硬件和驱动状态




# 可以用到的GPU资源

1. 如果自己的设备有NVIDIA GPU
    - 在自己的设备上安装cuda和cudnn
    - 安装对应版本的pytorch
    - https://pytorch.org/get-started/locally/

2. 上机时段，可以使用CFFF校内eGPU
    - 每人0.1卡A100
    - 预装notebook环境，浏览器访问

3. Google Colab
    - 预装notebook环境，浏览器访问
    - 免费提供T4 GPU
    - 需要良好网络资源，同时下载数据集也会更便捷

4. 阿里云
    1. 推荐：交互式建模 PAI-DSW
        - 可以免费试用250计算时*3个月
        - 独占高性能卡，A10/V100/G6
        - 方便快捷：预置镜像，notebook环境
        - https://free.aliyun.com/?productCode=learn
        
    2. CFFF平台-大学生300元代金券
        - 可以五折使用阿里云部分服务
        - 比较麻烦：需要手动配置服务器（镜像、公网ip、用户、python环境等）
        - https://university.aliyun.com/
    
    - 开通方法可参考飞书文档 https://fudannlp.feishu.cn/docx/IHYpdUcUsoBSGYxIxXacMfqOnXg

